# Clustering de Canciones

Proyecto de clustering listo para **Google Colab**.

**Pregunta:** ¿podemos agrupar canciones según características de audio?

**Método:** K-Means.

**Stack:** pandas, seaborn, scikit-learn

Dataset de ejemplo para clustering de canciones.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar / simular datos

In [ ]:
np.random.seed(RANDOM_STATE)
n = 300
df = pd.DataFrame({
    "danceability": np.random.uniform(0.2, 0.95, n).round(3),
    "energy": np.random.uniform(0.1, 0.98, n).round(3),
    "valence": np.random.uniform(0.05, 0.95, n).round(3),
    "tempo": np.random.uniform(60, 180, n).round(1),
    "acousticness": np.random.uniform(0.01, 0.9, n).round(3),
})
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print(df.describe().T)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df["danceability"], kde=True, ax=axes[0,0])
sns.histplot(df["energy"], kde=True, ax=axes[0,1])
sns.histplot(df["valence"], kde=True, ax=axes[1,0])
sns.histplot(df["tempo"], kde=True, ax=axes[1,1])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.scatterplot(data=df, x="danceability", y="energy")
plt.title("Danceability vs Energy")
plt.show()

### Hallazgos
- Features de audio típicos para clustering de géneros/moods.
- Ideal para segmentar canciones.

## 3. Preprocesamiento

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
print(X_scaled.shape)

## 4. Elegir k (Elbow + Silhouette)

In [ ]:
inertias = []
silhouettes = []
K = range(2, 8)

for k in K:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K, inertias, marker="o")
axes[0].set_title("Elbow")
axes[1].plot(K, silhouettes, marker="o", color="green")
axes[1].set_title("Silhouette")
plt.tight_layout()
plt.show()

## 5. Clustering final

In [ ]:
k_final = 4
km = KMeans(n_clusters=k_final, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = km.fit_predict(X_scaled)
print(df["cluster"].value_counts())

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="danceability", y="energy", hue="cluster", palette="tab10")
plt.title("Clusters de canciones")
plt.show()

## 6. Perfil de segmentos

In [ ]:
perfil = df.groupby("cluster").mean().round(3)
perfil["n"] = df["cluster"].value_counts()
perfil

## 7. Conclusiones

- Segmentos típicos: alta energía/dance, acústicas, etc.
- k=4 suele capturar bien los moods.
- Se puede extender a datasets reales de Spotify.

**Siguiente paso:** nombrar los clusters según género/mood.